# Exploración de resultados

Lee los archivos de `results/` del repositorio, así que no necesita datos, pesos ni GPU. En Colab, clona el repositorio y ejecuta `%cd` en la carpeta del proyecto antes de empezar. Reproduce la comparación de modelos, la concordancia por persona y tiempo, y el análisis de errores del artículo.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
RES = Path('results')
leer = lambda *p: json.loads(RES.joinpath(*p).read_text(encoding='utf-8'))

## 1. Comparación de modelos
Prueba propia y FLORES en chrF++; concordancia en % de acierto con verbos que el modelo no vio al entrenar (semilla 0).

In [ ]:
MODELOS = {'Sin ajustar': ('baseline_zeroshot', 'zeroshot_devtest', 'zeroshot'),
           'Principal': ('principal_s0', 'principal_s0_def_devtest', 'principal_s0'),
           'Sin 15 % al azar': ('exp_aleatorio15_s0', 'exp_aleatorio15_s0_def_devtest', 'exp_aleatorio15_s0'),
           'Sin 15 % peor puntuado': ('exp_filtrado15_s0', 'exp_filtrado15_s0_def_devtest', 'exp_filtrado15_s0'),
           'Retrotrad. >= 30': ('exp_bt30_s0', 'exp_bt30_s0_def_devtest', 'exp_bt30_s0'),
           'Retrotrad. >= 40': ('exp_bt40_s0', 'exp_bt40_s0_def_devtest', 'exp_bt40_s0'),
           'Normalizado 2k': ('exp_norm2k_s0', 'exp_norm2k_s0_def_devtest', 'exp_norm2k_s0'),
           'Normalizado 15k (final)': ('exp_norm15k_s0', 'exp_norm15k_s0_def_devtest', 'exp_norm15k_s0'),
           'Normalizado 15k + central': ('exp_norm15k_central_s0', 'exp_norm15k_central_s0_def_devtest', 'exp_norm15k_central_s0')}
filas = []
for nombre, (carpeta, flores, conc) in MODELOS.items():
    t, f, c = leer(carpeta, 'test.json'), leer('flores', flores + '.json'), leer('concordancia', conc + '.json')['acierto']
    filas.append({'Modelo': nombre, 'Prueba es→qu': t['es->qu']['chrF++'], 'Prueba qu→es': t['qu->es']['chrF++'],
                  'FLORES es→qu': f['es->qu']['chrF++'], 'FLORES qu→es': f['qu->es']['chrF++'],
                  'Concord. es→qu': c['es->qu'], 'Concord. qu→es': c['qu->es']})
tabla = pd.DataFrame(filas).set_index('Modelo')
tabla

In [ ]:
ax = tabla[['Concord. es→qu', 'Concord. qu→es']].plot.barh(figsize=(7, 4))
ax.set_xlabel('Acierto en el examen de concordancia (%)'); plt.tight_layout(); plt.show()

En la prueba propia los modelos quedan a menos de un punto entre sí; lo que los separa es la concordancia. El ciclo automático (`auto_it1`) queda fuera de la tabla porque guarda FLORES en `results/auto/`; ver el artículo.

## 2. Concordancia por persona y tiempo
Principal frente al modelo final. Cada celda tiene entre 10 y 35 casos, así que solo el patrón general es fiable.

In [ ]:
claves = ['1s', '2s', '3s', '1pi', '1pe', '2p', '3p', 'presente', 'pasado', 'futuro']
a, b = leer('concordancia', 'principal_s0.json')['acierto'], leer('concordancia', 'exp_norm15k_s0.json')['acierto']
desglose = pd.DataFrame({f'{d} {m}': [x[f'{d}:{k}'] for k in claves] for d in ('es->qu', 'qu->es') for m, x in (('principal', a), ('final', b))}, index=claves)
desglose

## 3. Análisis de errores
Las 8 traducciones con menor chrF++ del modelo final en es→qu.

In [ ]:
from sacrebleu.metrics import CHRF
pred = pd.read_csv(RES / 'exp_norm15k_s0' / 'test_predicciones.tsv', sep='	', keep_default_na=False)
eq = pred[pred.direccion == 'es->qu'].copy()
eq['chrF++'] = [CHRF(word_order=2).sentence_score(h, [r]).score for h, r in zip(eq.hip, eq.ref)]
eq[eq.src.str.split().str.len() >= 4].nsmallest(8, 'chrF++')[['src', 'ref', 'hip', 'chrF++']].round(1)

Entre los peores casos hay pares mal alineados (el quechua de una oración junto a la traducción de otra), encabezados y repeticiones. El artículo los cuantifica.

## 4. El apóstrofo
Las consonantes eyectivas se escriben con apóstrofo dentro de la palabra (p'unchaw, ch'aki). En los datos casi no aparece.

In [ ]:
d = leer('apostrofo_datos.json')
pd.DataFrame({k: {'frases con apóstrofo': f"{v['frases_con']} de {v['frases']}", 'palabras con apóstrofo': f"{v['palabras_con']} de {v['palabras']}", 'formas': v['formas']}
              for k, v in d.items()}).T

In [ ]:
p = leer('prueba_apostrofo.json')
# palabras del léxico con la glosa correcta, según cómo se escriba el apóstrofo
claves = ['recto \'', 'tipográfico ’', 'sin apóstrofo']
{k: f"{p[k]['aciertos']} de {p[k]['n']}" for k in claves}
